# Day 8 — Counterfactual Engine (Layer 7)

Turn predictions into **decisions**: what should a campaign change to raise its funding
probability? We perturb the **real controllable features** (goal, duration, launch timing,
title) through the actual Day-4 calibrated model, back each change with real historical
evidence, and rank + explain the improvements. Logic in `src/counterfactual/`.

**Honesty:** only mutable features (never country/currency); fabricated levers (revenue,
retention, founder…) deferred; cumulative probability **re-simulated jointly, not summed**;
model-simulated estimate, **not** a causal guarantee.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# One warm stack (calibrated model + Day-7 evidence frame). No faiss/torch -> no OpenMP issue.
from src.counterfactual.report_generator import ReportGenerator
rg = ReportGenerator(); pipe = rg.pipeline
gen, sim, ev, rank, exp = pipe.generator, pipe.simulator, pipe.evidence, pipe.ranking, pipe.explainer

# An improvable test campaign (high goal, long duration -> room to optimize).
df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][(df.iloc[ti]["main_category"]=="Film & Video") &
                  (df.iloc[ti]["usd_goal_real"].between(20000,40000))].sample(1, random_state=4).iloc[0].to_dict()
base_p = sim.baseline(row)
print("CAMPAIGN: %s | %s | goal $%.0f | baseline funding prob = %.3f" %
      (row["name"], row["main_category"], row["usd_goal_real"], base_p))

CAMPAIGN: Reel Kids | Film & Video | goal $38000 | baseline funding prob = 0.035


## Phase A — Understanding Counterfactual AI

Prediction says *what will happen*; a counterfactual says *what to change*. We perturb a raw
campaign and re-run the real model. Key rigor: cumulative effects are **re-simulated jointly**,
not summed — features interact and probabilities saturate. Notes:
`docs/day8_phaseA_counterfactual_notes.md`.

## Phase B — Candidate Generator (mutable features only)

Feasible changes to controllable features, targets seeded by Day-7 patterns (goal → category
funded-median). Country/currency are **structurally barred**.

In [2]:
cands = gen.generate(row)
pd.DataFrame([{"feature": c.feature, "current": c.current, "proposed": c.proposed, "tier": c.tier}
             for c in cands])

,feature,current,proposed,tier
0,goal,38000.0,5000.0,strong
1,goal,38000.0,19000.0,moderate
2,campaign_duration,59,15,moderate
3,campaign_duration,59,30,moderate
4,launch_weekday,Thursday,Tuesday,suggestive
5,title,no number,has number,suggestive


## Phase C — Probability Simulator (perturb → re-run model)

Single-feature Δp (independent) vs the honest greedy plan (cumulative re-simulated). Note the
plan total is **not** the sum of individual deltas.

In [3]:
sims = sim.simulate_all(row, cands)
print("Single-feature deltas:")
for s in sims:
    print("  %-16s %-16s Δ %+.3f -> %.3f" % (s["feature"], "%s→%s"%(s["current"],s["proposed"]), s["delta"], s["new_p"]))
plan = sim.greedy_plan(row, cands)
possum = sum(s["delta"] for s in sims if s["delta"] > 0)
print("\nsum of positive deltas: %+.3f   vs   re-simulated plan gain: %+.3f   (interaction gap %+.3f)"
      % (possum, plan["total_gain"], possum - plan["total_gain"]))

Single-feature deltas:
  campaign_duration 59→30            Δ +0.193 -> 0.228
  campaign_duration 59→15            Δ +0.052 -> 0.087
  goal             38000.0→5000.0   Δ +0.045 -> 0.080
  goal             38000.0→19000.0  Δ +0.019 -> 0.054
  launch_weekday   Thursday→Tuesday Δ +0.003 -> 0.038
  title            no number→has number Δ -0.001 -> 0.034



sum of positive deltas: +0.312   vs   re-simulated plan gain: +0.340   (interaction gap -0.028)


## Phase D — Evidence Validator (real historical support)

Each change is backed by the actual funding rate of matching past campaigns — no fabricated
counts.

In [4]:
validated = ev.validate_all(sims, row)
for v in validated[:3]:
    print("•", v["evidence"]["statement"])

• Film & Video campaigns with a 16-30d campaign funded at 44% vs a 40% baseline (lift 1.10, n=26,194).
• Film & Video campaigns with a <=15d campaign funded at 55% vs a 40% baseline (lift 1.36, n=2,746).
• Film & Video campaigns asking goal <= $5,000 funded at 51% vs a 40% baseline (lift 1.27, n=20,767). Funded campaigns here had a median goal of $5,000 vs $10,000 for failed.


## Phase E — Ranking Engine

Score = gain × tier × evidence × feasibility; drop negligible/negative; confidence label.

In [5]:
ranked = rank.rank(validated)
pd.DataFrame([{"priority": r["priority"], "feature": r["feature"], "Δp": r["delta"],
               "confidence": r["confidence"], "score": r["score"]} for r in ranked])

,priority,feature,Δp,confidence,score
0,1,campaign_duration,0.1932,medium,0.18048
1,2,campaign_duration,0.0518,high,0.05979
2,3,goal,0.0450,high,0.05706
3,4,goal,0.0191,medium,0.01917


## Phase F — Explanation Generator

Founder-readable action + model estimate + evidence + mandatory honesty caption.

In [6]:
for r in exp.explain_all(ranked)[:2]:
    print("•", r["explanation"], "\n")

• Shorten your campaign from 59 to 30 days. The model estimates your funding probability would rise from 3% to 23% (+19 points). Film & Video campaigns with a 16-30d campaign funded at 44% vs a 40% baseline (lift 1.10, n=26,194). Confidence: medium. Note: this is the model's estimate based on historical association, not a guarantee. 

• Shorten your campaign from 59 to 15 days. The model estimates your funding probability would rise from 3% to 9% (+5 points). Film & Video campaigns with a <=15d campaign funded at 55% vs a 40% baseline (lift 1.36, n=2,746). Confidence: high. Note: this is the model's estimate based on historical association, not a guarantee. 



## Phase G — Counterfactual Pipeline

One call: raw campaign → ranked recommendations + optimization plan (menu + path).

In [7]:
report = rg.generate(row)   # runs the full pipeline
plan = report["optimization_plan"]
print("baseline %.0f%% -> final %.0f%% (total %+.0f points)" %
      (plan["baseline_p"]*100, plan["final_p"]*100, plan["total_gain"]*100))
for s in plan["steps"]:
    print("  → %s  ⇒ %.0f%% (%+.0f)" % (s["action"], s["cumulative_p"]*100, s["marginal_gain"]*100))

baseline 3% -> final 38% (total +34 points)
  → Shorten your campaign from 59 to 30 days  ⇒ 23% (+19)
  → Lower your funding goal from $38,000 to $5,000  ⇒ 38% (+15)


## Phase H — Startup Optimization Report

The founder-facing report — current probability, ranked improvements, cumulative path,
optimized profile (mutable features only), and an overall recommendation.

In [8]:
print(rg.render_text(report))
print("\nOptimized profile:", {k: report["optimized_profile"][k]
      for k in ["goal_usd","campaign_duration_days","estimated_funding_probability"]})

STARTUP OPTIMIZATION REPORT — Reel Kids
  Film & Video | goal $38,000
  Current funding probability: 3%
  ------------------------------------------------------------
  RECOMMENDED IMPROVEMENTS (ranked):
   1. [medium] Shorten your campaign from 59 to 30 days (→ 23%, +19%)
   2. [high] Shorten your campaign from 59 to 15 days (→ 9%, +5%)
   3. [high] Lower your funding goal from $38,000 to $5,000 (→ 8%, +4%)
   4. [medium] Lower your funding goal from $38,000 to $19,000 (→ 5%, +2%)
  ------------------------------------------------------------
  OPTIMIZATION PATH (cumulative): 3%
   → Shorten your campaign from 59 to 30 days ⇒ 23% (+19%)
   → Lower your funding goal from $38,000 to $5,000 ⇒ 38% (+15%)
   FINAL: 38% (total +34%)
  ------------------------------------------------------------
  OVERALL: Prioritize: shorten your campaign from 59 to 30 days. Applying all 2 recommended change(s), the model estimates your funding probability could rise from 3% to 38% (+34 points).
  Note: Mod

## Phase I — Serialization

The engine is a configuration over the Day-4 model + Day-7 patterns; persisted as JSON with a
content hash + dependency hashes, `verify()` detects drift.

In [9]:
from src.counterfactual.serialization import CounterfactualSerializer
ser = CounterfactualSerializer().build(); ser.save()
print(ser.describe())
print("verify:", ser.verify()["all_ok"])

Counterfactual engine v1.0.0 (content a16c107361bc…): mutable ['goal', 'campaign_duration', 'launch_weekday', 'title'], immutable ['country', 'currency']; 3 deps hashed, 11 features deferred.
verify: True


## Day 8 complete — Layer 7 ✅

Mutable-only candidates (Day-7-seeded) → perturb-through-real-model simulator (greedy
re-simulated plan) → real-evidence validator → ranking → honesty-captioned explanations →
pipeline → optimization report → content-hashed serialization. The recruiter-demo feature,
done truthfully: **3% → 38% by shortening the campaign and lowering the goal** — a
model-simulated estimate grounded in historical patterns, not a guarantee.